#REGRESION LINEAL MULTIPLE DE POLINOMIO DE PRIMER GRADO


Este código implementa un modelo de regresión lineal múltiple de primer grado con el objetivo de analizar la relación entre variables climáticas y la producción agrícola. A partir de una base de datos previamente depurada, se selecciona una variable dependiente (como superficie cosechada o sembrada en una zona específica) y un conjunto de variables independientes relacionadas con el clima.

El modelo se construye utilizando el método de mínimos cuadrados ordinarios (OLS), lo que permite estimar qué tanto influyen las variables climáticas en el comportamiento de la variable objetivo. Posteriormente, se evalúa el desempeño del modelo mediante un resumen estadístico y pruebas básicas de validación, con el fin de verificar que los resultados sean consistentes y confiables.

In [1]:
# 1) CARGAR LIBRERÍAS

import pandas as pd  # Para manipular los datos en tablas (DataFrame)
import statsmodels.api as sm  # Para construir y analizar el modelo de regresión
import matplotlib.pyplot as plt  # Para graficar resultados
import statsmodels.stats.api as sms  # Para realizar pruebas estadísticas sobre el modelo (ej. heterocedasticidad)


In [2]:
# 2) SUBIR ARCHIVO

from google.colab import files  # Permite subir archivos desde tu computadora

uploaded = files.upload()  # Abre la ventana para seleccionar el archivo
archivo = list(uploaded.keys())[0]  # Obtiene el nombre del archivo cargado

Saving TABLA CONJUNTA .csv to TABLA CONJUNTA .csv


El archivo que se debe cargar es un archivo en formato CSV (.csv) que contiene
todas las variables climáticas y agrícolas por zona. Esto incluye variables como
evaporación, lluvia y temperatura para distintas zonas (por ejemplo: La Barca,
Jamay, Ocotlán, etc.), así como las columnas de superficie sembrada y cosechada
para cada una de estas zonas.

La base de datos ya se encuentra previamente depurada y acotada al periodo de
2003 a 2024. Este recorte se realizó con el fin de homologar la información y
poder relacionar correctamente las variables mediante la llave común del año,
facilitando así el análisis conjunto entre datos climáticos y de producción.

In [3]:
# 3) LEER BASE DE DATOS

df = pd.read_csv(archivo)  # Carga el archivo CSV como DataFrame

# Ver primeras filas para confirmar que se cargó correctamente
print(df.head())

    AÑO  EVAPORACION LA BARCA   EVAPORACION JAMAY  EVAPORACION OCOTLAN  \
0  2003              131.90100         148.124000           156.121708   
1  2004              135.13450         151.439441           136.129292   
2  2005              151.41125         156.955167           157.909542   
3  2006              155.83000         155.155000           142.895833   
4  2007              143.78250         147.241941           164.998083   

   EVAPORACION BRISEÑAS  LLUVIA LA BARCA  LLUVIA JAMAY  LLUVIA OCOTLAN  \
0            164.138893        83.733333     51.394583       61.321667   
1            162.336662        89.888333     83.124782       89.691000   
2            173.203012        63.000000     43.633333       54.152875   
3            167.569167        67.458500     66.858333       53.792917   
4            166.331644        63.920000     61.891991       75.491667   

   LLUVIA BRISEÑAS   TMaxE LA BARCA  ...  TMinP OCOTLAN  TMinP BRISEÑAS  \
0         61.336751       32.541667

In [4]:
print(df.columns.tolist())  # Muestra la lista de nombres de todas las columnas del DataFrame

['AÑO', 'EVAPORACION LA BARCA ', 'EVAPORACION JAMAY', 'EVAPORACION OCOTLAN', 'EVAPORACION BRISEÑAS', 'LLUVIA LA BARCA', 'LLUVIA JAMAY', 'LLUVIA OCOTLAN', 'LLUVIA BRISEÑAS ', 'TMaxE LA BARCA', 'TMaxE JAMAY', 'TMaxE OCOTLAN', 'TMaxE BRISEÑAS', 'TMaxP LA BARCA ', 'TMaxP JAMAY', 'TMaxP OCOTLAN', 'TMaxP BRISEÑAS', 'TMedP LA BARCA', 'TMedP JAMAY', 'TMedP OCOTLAN', 'TMedP BRISEÑAS', 'TMinE LA BARCA', 'TMinE JAMAY', 'TMinE OCOTLAN', 'TMinE BRISEÑAS', 'TMinP LA BARCA', 'TMinP JAMAY', 'TMinP OCOTLAN', 'TMinP BRISEÑAS', 'Sembrada LA BARCA', 'Sembrada JAMAY', 'Sembrada OCOTLAN', 'Sembrada BRISEÑAS', 'Cosechada LA BARCA', 'Cosechada JAMAY', 'Cosechada OCOTLAN ', 'Cosechada BRISEÑAS']


In [5]:
#4) Limpieza básica de nombres de columnas

df.columns = df.columns.str.strip()  # Elimina espacios en blanco al inicio o final de los nombres de columnas#

Se realizó una limpieza básica en los nombres de las columnas con el fin de asegurar que coincidieran correctamente con las variables que se utilizarán en el modelo. En particular, se eliminaron espacios en blanco innecesarios que podrían generar errores al momento de llamar o manipular las variables dentro del código.

In [6]:
print(df.columns.tolist())  # Vuelve a mostrar los nombres de columnas para confirmar que la limpieza se aplicó correctamente

['AÑO', 'EVAPORACION LA BARCA', 'EVAPORACION JAMAY', 'EVAPORACION OCOTLAN', 'EVAPORACION BRISEÑAS', 'LLUVIA LA BARCA', 'LLUVIA JAMAY', 'LLUVIA OCOTLAN', 'LLUVIA BRISEÑAS', 'TMaxE LA BARCA', 'TMaxE JAMAY', 'TMaxE OCOTLAN', 'TMaxE BRISEÑAS', 'TMaxP LA BARCA', 'TMaxP JAMAY', 'TMaxP OCOTLAN', 'TMaxP BRISEÑAS', 'TMedP LA BARCA', 'TMedP JAMAY', 'TMedP OCOTLAN', 'TMedP BRISEÑAS', 'TMinE LA BARCA', 'TMinE JAMAY', 'TMinE OCOTLAN', 'TMinE BRISEÑAS', 'TMinP LA BARCA', 'TMinP JAMAY', 'TMinP OCOTLAN', 'TMinP BRISEÑAS', 'Sembrada LA BARCA', 'Sembrada JAMAY', 'Sembrada OCOTLAN', 'Sembrada BRISEÑAS', 'Cosechada LA BARCA', 'Cosechada JAMAY', 'Cosechada OCOTLAN', 'Cosechada BRISEÑAS']


In [7]:
# 5) Definir variables del modelo

# Variable dependiente (Y)
# Corresponde a la variable que se quiere explicar (producción de maíz)
y = df["Cosechada OCOTLAN"]

# Variables independientes (X)
X = df[[
    #"EVAPORACION LA BARCA",
    #"EVAPORACION JAMAY",
    #"EVAPORACION OCOTLAN",
    #"EVAPORACION BRISEÑAS",

    #"LLUVIA LA BARCA",
    #"LLUVIA JAMAY",
    #"LLUVIA OCOTLAN",
    #"LLUVIA BRISEÑAS",

    #"TMaxE LA BARCA",
    #"TMaxE JAMAY",
    #"TMaxE OCOTLAN",
    #"TMaxE BRISEÑAS",

    #"TMaxP LA BARCA",
    #"TMaxP JAMAY",
    #"TMaxP OCOTLAN",
    #"TMaxP BRISEÑAS",

    #"TMedP LA BARCA",
    #"TMedP JAMAY",
    #"TMedP OCOTLAN",
    #"TMedP BRISEÑAS",

    #"TMinE LA BARCA",
    #"TMinE JAMAY",
    #"TMinE OCOTLAN",
    #"TMinE BRISEÑAS",

    #"TMinP LA BARCA",
    #"TMinP JAMAY",
    #"TMinP OCOTLAN",
    #"TMinP BRISEÑAS"
]]

En la definición de la variable dependiente (Y) se puede seleccionar cualquiera de las variables agrícolas disponibles en la base de datos, como: Cosechada o Sembrada para cada zona (La Barca, Jamay, Ocotlán o Briseñas). Esto permite analizar diferentes combinaciones dependiendo del enfoque del modelo.
Por otro lado, en la definición de las variables independientes (X), se pueden activar o desactivar variables climáticas simplemente quitando o agregando comentarios en cada línea. Esto se hizo con el objetivo de no tener que reescribir todas las variables desde cero, facilitando la construcción de distintos modelos y la comparación entre resultados.

In [8]:
# 6) Agregar constante al modelo

X = sm.add_constant(X)  # Añade el término independiente necesario para la regresión

Se agrega una constante al modelo para incluir el término independiente (intercepto), el cual representa el valor base de la variable dependiente cuando todas las variables independientes son iguales a cero. Esto es necesario para que el modelo no esté forzado a pasar por el origen y pueda ajustarse de manera más realista a los datos.

In [10]:
# 7) Ajustar modelo de regresión lineal múltiple

modelo = sm.OLS(y, X).fit()  # Construye y ajusta el modelo usando mínimos cuadrados ordinarios (OLS)

Esta función construye y ajusta un modelo de regresión lineal múltiple utilizando el método de mínimos cuadrados ordinarios (OLS). A partir de las variables independientes (X) y la variable dependiente (Y), el modelo estima los coeficientes que mejor explican la relación entre ellas, minimizando el error entre los valores reales y los valores predichos.

In [11]:
# 8) Resultados de la regresion lineal de primer grado
print(modelo.summary())  # Muestra un resumen completo del modelo (coeficientes, R², p-values, etc.)

                            OLS Regression Results                            
Dep. Variable:      Cosechada OCOTLAN   R-squared:                       0.075
Model:                            OLS   Adj. R-squared:                  0.029
Method:                 Least Squares   F-statistic:                     1.619
Date:                Wed, 29 Apr 2026   Prob (F-statistic):              0.218
Time:                        22:25:39   Log-Likelihood:                -203.31
No. Observations:                  22   AIC:                             410.6
Df Residuals:                      20   BIC:                             412.8
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                    coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------
const          1.532e+04   5703.540      2.687

In [16]:
# Validación del modelo con pruebas estadísticas

from scipy.stats import shapiro  # Prueba de normalidad
from statsmodels.stats.stattools import durbin_watson  # Prueba de autocorrelación

# Prueba de normalidad (Shapiro-Wilk)
stat, p_shapiro = shapiro(modelo.resid)  # Evalúa si los errores del modelo siguen una distribución normal

# Prueba de heterocedasticidad (Breusch-Pagan)
test = sms.het_breuschpagan(modelo.resid, X)  # Evalúa si la varianza de los errores es constante
p_bp = test[1]  # p-value de la prueba

# Prueba de autocorrelación (Durbin-Watson)
dw = durbin_watson(modelo.resid)  # Evalúa si existe correlación entre los errores

# Resultados de las pruebas
print(f"Shapiro-Wilk (normalidad): {p_shapiro}")  # p > 0.05 sugiere normalidad
print(f"Durbin-Watson (autocorrelación): {dw}")  # ≈ 2 indica independencia
print(f"Breusch-Pagan (heterocedasticidad): {p_bp}")  # p > 0.05 sugiere varianza constante

Shapiro-Wilk (normalidad): 0.07439632375454723
Durbin-Watson (autocorrelación): 0.5345556487964171
Breusch-Pagan (heterocedasticidad): 0.1662953503549015


Finalmente, se aplicaron tres pruebas estadísticas para validar el modelo. La prueba de Shapiro-Wilk permite verificar si los errores siguen una distribución normal, Durbin-Watson evalúa si existe autocorrelación entre los errores, y Breusch-Pagan analiza si la varianza de los errores es constante. Estas pruebas ayudan a confirmar que el modelo cumple con los supuestos básicos de la regresión lineal y que sus resultados son confiables.